# w6_cpu3_o2_hetero_graph_pinns_and_dut_v2


In [ ]:
import hashlib, os, subprocess, sys, tempfile, zipfile
from pathlib import Path

WORK, MOUNT = Path('/kaggle/working'), Path('/kaggle/input')
assert WORK.is_dir(), 'Run on Kaggle'
REV = '92e6e0ce78b1bf47d08e7b0464109751bce6acb7'
with tempfile.TemporaryDirectory(prefix='w6_gr_') as ts:
    tmp = Path(ts)
    repo, meta_dir = tmp / 'repo', tmp / 'pinns_meta'
    subprocess.run(['git', 'init', '-q', str(repo)], check=True)
    subprocess.run(['git', '-C', str(repo), 'fetch', '-q', '--depth=1', 'https://github.com/pjyyyds/Unstructured-ShortPose-Pred.git', REV], check=True)
    subprocess.run(['git', '-C', str(repo), 'checkout', '-q', '--detach', 'FETCH_HEAD'], check=True)
    env = {**os.environ, 'PYTHONPATH': str(repo), 'PYTHONDONTWRITEBYTECODE': '1', 'CUDA_VISIBLE_DEVICES': ''}
    heldout_bin = next(MOUNT.rglob('shortpose_heldout.bin'))
    pose_npz = tmp / 'pose.npz'
    with zipfile.ZipFile(heldout_bin) as zf:
        pose_npz.write_bytes(zf.read('pose.npz'))
    z_hits = list(MOUNT.rglob('dut_shared_space_graph_transfer.zip'))
    if z_hits:
        dut_zip = z_hits[0]
    else:
        pg_f = next(MOUNT.rglob('private_graph.npz'))
        et_f = next(MOUNT.rglob('errors_and_targets.npz'))
        dut_zip = tmp / 'dut_graph.zip'
        with zipfile.ZipFile(dut_zip, 'w', zipfile.ZIP_DEFLATED) as dz:
            dz.write(pg_f, 'private_graph.npz')
            dz.write(et_f, 'errors_and_targets.npz')
    subprocess.run([sys.executable, '-m', 'scripts.stage_pinns_metadata', '--output-dir', str(meta_dir)], cwd=repo, env=env, check=True)
    dest = WORK / 'o2_hetero_graph_pinns_and_dut_5seed.zip'
    subprocess.run([sys.executable, '-u', '-m', 'scripts.run_o2_hetero_graph_pinns_and_dut', str(meta_dir), str(pose_npz), str(dut_zip), str(dest)], cwd=repo, env=env, check=True)
    print('DONE:', dest, dest.stat().st_size, hashlib.sha256(dest.read_bytes()).hexdigest(), flush=True)
